# Transformer from Scratch

**Lecture 2, Part 3 — Vizuara Modern Robot Learning Bootcamp**

We build a complete **Transformer** in ~100 lines of PyTorch, piece by piece:

| Component | What it does | Lines of code |
|-----------|-------------|---------------|
| Self-Attention | Every word looks at every other word | 8 lines |
| Multi-Head Attention | 4 parallel attention "perspectives" | 15 lines |
| Positional Encoding | Inject word order (sinusoidal) | 10 lines |
| Transformer Block | Attention + FFN + LayerNorm + residual | 12 lines |
| Full Transformer | Stack N blocks into a sequence encoder | 10 lines |

By the end, you'll see exactly **why Transformers fix the three GRU failures** from Part 2:
1. **Bottleneck** — Transformer outputs N vectors (one per token), not one
2. **No cross-modal talk** — put vision + text in the same sequence, attention connects them
3. **Sequential processing** — attention computes all pairs simultaneously

---

In [ ]:
!pip install -q torch numpy matplotlib 2>&1 | tail -3

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

# Vizuara color palette
ACCENT = '#D97757'
BLUE   = '#5B8CB8'
TEAL   = '#7DA488'
PURPLE = '#9B7EC8'
WARM   = '#C4956A'
BG     = '#FAF8F5'

plt.rcParams['figure.facecolor'] = BG
plt.rcParams['axes.facecolor'] = BG
plt.rcParams['font.size'] = 12

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## Our Running Example

We'll trace attention through a robot instruction. Same sentences as the Language Encoding notebook — so you can directly compare GRU vs Transformer.

In [ ]:
sentence = "The robot picked up the red cup from the table"
words = sentence.lower().split()

# Build a small vocabulary
vocab_words = sorted(set(words))
vocab = {w: i for i, w in enumerate(vocab_words)}
VOCAB_SIZE = len(vocab)
SEQ_LEN = len(words)
D_MODEL = 128  # embedding dimension

print(f"Sentence: '{sentence}'")
print(f"Words ({SEQ_LEN}): {words}")
print(f"Vocabulary ({VOCAB_SIZE}): {vocab}")

# Tokenize
token_ids = torch.tensor([[vocab[w] for w in words]])  # [1, seq_len]
print(f"\nToken IDs: {token_ids[0].tolist()}")

In [ ]:
# Create word embeddings (random initialization — like before training)
embedding = nn.Embedding(VOCAB_SIZE, D_MODEL)

x = embedding(token_ids)  # [1, seq_len, d_model]
x = x.squeeze(0)          # [seq_len, d_model] — work with single sentence

print(f"Input shape: {list(x.shape)}")
print(f"  → {SEQ_LEN} words, each represented as a {D_MODEL}-d vector")
print(f"  → Right now these are random. Training would make them meaningful.")

---
## Step 1: Self-Attention — The Core Mechanism

Each word gets projected into three vectors:

| Vector | Name | Purpose | Analogy |
|--------|------|---------|--------|
| **Q** | Query | "What am I looking for?" | A search query you type |
| **K** | Key | "What do I contain?" | A page title / tag |
| **V** | Value | "Here's my actual content" | The page content itself |

Then: **score = Q · K^T** (who should I attend to?) → **softmax** → **weighted sum of V** (mix their content).

In [ ]:
# Step 1a: Create Q, K, V projections
d_k = D_MODEL  # key/query dimension (= d_model for single-head)

W_q = nn.Linear(D_MODEL, d_k, bias=False)
W_k = nn.Linear(D_MODEL, d_k, bias=False)
W_v = nn.Linear(D_MODEL, d_k, bias=False)

with torch.no_grad():
    Q = W_q(x)  # [seq_len, d_k]
    K = W_k(x)  # [seq_len, d_k]
    V = W_v(x)  # [seq_len, d_k]

print(f"Q shape: {list(Q.shape)}  — each word asks 'what am I looking for?'")
print(f"K shape: {list(K.shape)}  — each word says 'here's how to find me'")
print(f"V shape: {list(V.shape)}  — each word says 'here's my content'")
print(f"\nThree different 'views' of the same {SEQ_LEN} words.")

In [ ]:
# Step 1b: Compute attention scores (Q · K^T)
with torch.no_grad():
    scores_raw = Q @ K.T  # [seq_len, seq_len]

print(f"Raw attention scores shape: {list(scores_raw.shape)}")
print(f"  → {SEQ_LEN}x{SEQ_LEN} matrix: every word's query dotted with every word's key")
print(f"\nRaw scores (first 3 rows):")
for i in range(3):
    vals = ', '.join(f'{v:.1f}' for v in scores_raw[i].tolist())
    print(f"  '{words[i]}' → [{vals}]")
print(f"\nProblem: scores range from {scores_raw.min():.1f} to {scores_raw.max():.1f}")
print(f"With d_k={d_k}, dot products can get very large → softmax becomes one-hot!")

In [ ]:
# Step 1c: Scale by sqrt(d_k) — the temperature fix
with torch.no_grad():
    scores_scaled = scores_raw / math.sqrt(d_k)

print(f"Scale factor: sqrt({d_k}) = {math.sqrt(d_k):.1f}")
print(f"After scaling: range [{scores_scaled.min():.2f}, {scores_scaled.max():.2f}]")

# Compare softmax with and without scaling
weights_unscaled = F.softmax(scores_raw[0], dim=-1)
weights_scaled = F.softmax(scores_scaled[0], dim=-1)

fig, axes = plt.subplots(1, 2, figsize=(14, 3))

axes[0].bar(range(SEQ_LEN), weights_unscaled.numpy(), color=ACCENT)
axes[0].set_xticks(range(SEQ_LEN))
axes[0].set_xticklabels(words, rotation=45, ha='right', fontsize=9)
axes[0].set_title(f'WITHOUT scaling (d_k={d_k})', color=ACCENT, fontweight='bold')
axes[0].set_ylabel('Attention weight')
max_val = weights_unscaled.max().item()
axes[0].text(0.95, 0.9, f'max={max_val:.3f}\nAlmost one-hot!',
             transform=axes[0].transAxes, ha='right', va='top', fontsize=10, color=ACCENT)

axes[1].bar(range(SEQ_LEN), weights_scaled.numpy(), color=TEAL)
axes[1].set_xticks(range(SEQ_LEN))
axes[1].set_xticklabels(words, rotation=45, ha='right', fontsize=9)
axes[1].set_title(f'WITH scaling (÷ sqrt({d_k}))', color=TEAL, fontweight='bold')
axes[1].set_ylabel('Attention weight')
max_val2 = weights_scaled.max().item()
axes[1].text(0.95, 0.9, f'max={max_val2:.3f}\nSmooth distribution!',
             transform=axes[1].transAxes, ha='right', va='top', fontsize=10, color=TEAL)

plt.suptitle('Why Scaling Matters: "the" attends to all other words',
             fontsize=14, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

print("Without scaling: softmax becomes almost one-hot (only attends to 1 word).")
print("With scaling: smooth distribution — the model can attend to multiple words.")

In [ ]:
# Step 1d: Full self-attention — the complete formula
def self_attention(x, W_q, W_k, W_v):
    """Scaled dot-product self-attention.

    x: [seq_len, d_model]
    Returns: [seq_len, d_model] — each word enriched by attending to all others
    """
    Q = W_q(x)                           # [seq_len, d_k]
    K = W_k(x)                           # [seq_len, d_k]
    V = W_v(x)                           # [seq_len, d_v]

    d_k = Q.shape[-1]
    scores = Q @ K.T / math.sqrt(d_k)    # [seq_len, seq_len]
    weights = F.softmax(scores, dim=-1)   # each row sums to 1
    output = weights @ V                  # [seq_len, d_v]
    return output, weights

with torch.no_grad():
    out, attn_weights = self_attention(x, W_q, W_k, W_v)

print(f"Input:  {list(x.shape)}  — {SEQ_LEN} words × {D_MODEL}-d")
print(f"Output: {list(out.shape)}  — {SEQ_LEN} words × {D_MODEL}-d")
print(f"Attention weights: {list(attn_weights.shape)} — every word's attention over all words")
print(f"\nEach output word is now a WEIGHTED MIX of all input words' Values.")
print(f"'picked' knows about 'cup'. 'red' knows about 'cup'. Everyone knows everything.")

---
## Visualize: The Attention Map

The attention weight matrix shows **who attends to whom**. Each row is one word's attention distribution over all words.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(attn_weights.numpy(), cmap='YlOrRd', vmin=0)
ax.set_xticks(range(SEQ_LEN))
ax.set_yticks(range(SEQ_LEN))
ax.set_xticklabels(words, rotation=45, ha='right', fontsize=11)
ax.set_yticklabels(words, fontsize=11)
ax.set_xlabel('Attends TO (Key)', fontsize=12)
ax.set_ylabel('Attends FROM (Query)', fontsize=12)

for i in range(SEQ_LEN):
    for j in range(SEQ_LEN):
        val = attn_weights[i, j].item()
        color = 'white' if val > 0.15 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8, color=color)

plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Self-Attention Weights (random init)', fontsize=14, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

print("Each ROW shows one word's attention distribution (sums to 1.0).")
print("With random weights, patterns are meaningless — training makes them meaningful.")
print("After training: 'picked' would strongly attend to 'cup' and 'robot'.")

---
## Step 2: Multi-Head Attention — Different Perspectives

One attention head learns **one** relationship pattern (e.g., "what object?"). We need multiple heads to capture different patterns simultaneously:

| Head | What it might learn | Example |
|------|--------------------|---------|
| Head 1 | Subject–verb | "robot" ↔ "picked" |
| Head 2 | Adjective–noun | "red" ↔ "cup" |
| Head 3 | Verb–object | "picked" ↔ "cup" |
| Head 4 | Spatial relations | "from" ↔ "table" |

In [ ]:
class AttentionHead(nn.Module):
    """Single attention head: Q, K, V projections + scaled dot-product."""
    def __init__(self, d_model, d_head):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_head, bias=False)
        self.W_k = nn.Linear(d_model, d_head, bias=False)
        self.W_v = nn.Linear(d_model, d_head, bias=False)
        self.d_head = d_head

    def forward(self, x):
        Q = self.W_q(x)  # [seq_len, d_head]
        K = self.W_k(x)
        V = self.W_v(x)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_head)
        weights = F.softmax(scores, dim=-1)
        return weights @ V, weights


class MultiHeadAttention(nn.Module):
    """Multi-head attention: N parallel heads, then project back."""
    def __init__(self, d_model=128, n_heads=4):
        super().__init__()
        assert d_model % n_heads == 0
        d_head = d_model // n_heads
        self.heads = nn.ModuleList([
            AttentionHead(d_model, d_head) for _ in range(n_heads)
        ])
        self.proj = nn.Linear(d_model, d_model)  # combine heads

    def forward(self, x):
        head_outputs = []
        head_weights = []
        for h in self.heads:
            out, w = h(x)
            head_outputs.append(out)
            head_weights.append(w)
        concat = torch.cat(head_outputs, dim=-1)  # [seq_len, d_model]
        return self.proj(concat), head_weights


# Build and test
mha = MultiHeadAttention(d_model=D_MODEL, n_heads=4)

with torch.no_grad():
    mha_out, all_head_weights = mha(x)

print(f"Multi-Head Attention:")
print(f"  Input:  {list(x.shape)}")
print(f"  Output: {list(mha_out.shape)}")
print(f"  Heads:  4 × {D_MODEL // 4}-d each = {D_MODEL}-d total")
print(f"  Attention maps: {len(all_head_weights)} heads, each {list(all_head_weights[0].shape)}")

n_params = sum(p.numel() for p in mha.parameters())
print(f"  Parameters: {n_params:,}")

In [ ]:
# Visualize all 4 attention heads side by side
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
head_colors = [ACCENT, BLUE, TEAL, PURPLE]

for h_idx, (ax, weights) in enumerate(zip(axes, all_head_weights)):
    w = weights.detach().numpy()
    im = ax.imshow(w, cmap='YlOrRd', vmin=0, vmax=w.max())
    ax.set_xticks(range(SEQ_LEN))
    ax.set_yticks(range(SEQ_LEN))
    ax.set_xticklabels(words, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(words, fontsize=8)
    ax.set_title(f'Head {h_idx + 1} ({D_MODEL // 4}-d)',
                 fontsize=12, color=head_colors[h_idx], fontweight='bold')
    for i in range(SEQ_LEN):
        for j in range(SEQ_LEN):
            val = w[i, j]
            color = 'white' if val > 0.15 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=6, color=color)

plt.suptitle('4 Attention Heads — Each Learns Different Patterns',
             fontsize=15, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

print("Each head sees the same sentence but attends to different patterns.")
print("After training, Head 1 might learn subject-verb, Head 2 adjective-noun, etc.")
print("The final projection combines all 4 perspectives into one rich representation.")

---
## Step 3: Positional Encoding — Where's the Word Order?

Attention treats input as a **set**, not a sequence. Without position information:

> "The robot picked up the cup" = "cup the picked up robot the"

We add a unique sinusoidal signal to each position so the model knows word order.

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding (Vaswani et al., 2017)."""
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)  # even dims
        pe[:, 1::2] = torch.cos(position * div_term)  # odd dims
        self.register_buffer('pe', pe)

    def forward(self, seq_len):
        return self.pe[:seq_len]  # [seq_len, d_model]


pos_enc = PositionalEncoding(D_MODEL)
pe = pos_enc(SEQ_LEN)  # [seq_len, d_model]

print(f"Positional encoding shape: {list(pe.shape)}")
print(f"  → One unique {D_MODEL}-d fingerprint per position")
print(f"  → Added to word embeddings: x = embedding + pos_encoding")

In [ ]:
# Visualize positional encoding patterns
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: heatmap of positional encodings
pe_50 = PositionalEncoding(D_MODEL)(50).numpy()  # 50 positions
ax = axes[0]
im = ax.imshow(pe_50.T[:32, :], aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xlabel('Position', fontsize=11)
ax.set_ylabel('Encoding dimension (first 32)', fontsize=11)
ax.set_title('Sinusoidal Positional Encoding', fontsize=13, color=ACCENT, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)

# Right: cosine similarity between positions
pe_norm = pe_50 / np.linalg.norm(pe_50, axis=1, keepdims=True)
pos_sim = pe_norm @ pe_norm.T
ax = axes[1]
im = ax.imshow(pos_sim[:20, :20], cmap='RdYlGn', vmin=-0.5, vmax=1)
ax.set_xlabel('Position', fontsize=11)
ax.set_ylabel('Position', fontsize=11)
ax.set_title('Position Similarity (first 20)', fontsize=13, color=ACCENT, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

print("Left: Each row is a sinusoidal wave at a different frequency.")
print("  Low dims = slow oscillation (captures coarse position).")
print("  High dims = fast oscillation (captures fine position).")
print("\nRight: Nearby positions are similar, distant positions are different.")
print("  This is exactly what we want — 'cup' at position 6 ≠ 'cup' at position 1.")

In [ ]:
# Show that positional encoding makes word order matter
with torch.no_grad():
    x_no_pos = embedding(token_ids).squeeze(0)  # [seq_len, d_model]
    x_with_pos = x_no_pos + pos_enc(SEQ_LEN)     # add position info

    # Shuffle words ("cup the picked up robot the...")
    perm = torch.randperm(SEQ_LEN)
    x_shuffled_no_pos = x_no_pos[perm]
    x_shuffled_with_pos = embedding(token_ids).squeeze(0)[perm] + pos_enc(SEQ_LEN)

    # Compare: without pos encoding, shuffled attention = original attention
    _, weights_orig = self_attention(x_no_pos, W_q, W_k, W_v)
    _, weights_shuf = self_attention(x_shuffled_no_pos, W_q, W_k, W_v)

    # With pos encoding, they differ
    _, weights_orig_pos = self_attention(x_with_pos, W_q, W_k, W_v)
    _, weights_shuf_pos = self_attention(x_shuffled_with_pos, W_q, W_k, W_v)

shuffled_words = [words[i] for i in perm.tolist()]

diff_no_pos = (weights_orig - weights_shuf).abs().mean().item()
diff_with_pos = (weights_orig_pos - weights_shuf_pos).abs().mean().item()

print(f"Original:  {' '.join(words)}")
print(f"Shuffled:  {' '.join(shuffled_words)}")
print(f"\nMean absolute difference in attention weights:")
print(f"  WITHOUT positional encoding: {diff_no_pos:.6f}  ← Almost identical!")
print(f"  WITH positional encoding:    {diff_with_pos:.4f}  ← Different!")
print(f"\nWithout position info, attention is ORDER-BLIND (a set operation).")
print(f"Positional encoding fixes this.")

---
## Step 4: The Transformer Block

Self-attention is the core, but a full block adds:

1. **Multi-Head Attention** — who should I listen to?
2. **Feed-Forward Network (FFN)** — what should I think about what I heard? (expand to 4× width, then compress back)
3. **LayerNorm** — stabilize activations (Pre-LN style, used by GPT/PaLM/Gemma)
4. **Residual connections** — `x + sublayer(x)` — let gradients flow through deep networks

In [ ]:
class TransformerBlock(nn.Module):
    """One Transformer block: attention + FFN, with LayerNorm + residuals."""
    def __init__(self, d_model=128, n_heads=4, d_ff=512):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.norm1 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),   # expand to 4x
            nn.GELU(),                  # activation
            nn.Linear(d_ff, d_model),   # compress back
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # Attention sub-layer with residual
        attn_out, _ = self.attn(self.norm1(x))  # "Who should I listen to?"
        x = x + attn_out                         # residual connection

        # FFN sub-layer with residual
        x = x + self.ffn(self.norm2(x))          # "What should I make of it?"
        return x


block = TransformerBlock(d_model=D_MODEL, n_heads=4, d_ff=512)

with torch.no_grad():
    block_out = block(x)

print(f"Transformer Block:")
print(f"  Input:  {list(x.shape)}")
print(f"  Output: {list(block_out.shape)}")
n_params = sum(p.numel() for p in block.parameters())
print(f"  Parameters: {n_params:,}")
print(f"\nBreakdown:")
print(f"  Multi-Head Attention: {sum(p.numel() for p in block.attn.parameters()):,} params")
print(f"  FFN (expand 128→512→128): {sum(p.numel() for p in block.ffn.parameters()):,} params")
print(f"  LayerNorm ×2: {sum(p.numel() for p in block.norm1.parameters()) + sum(p.numel() for p in block.norm2.parameters()):,} params")

In [ ]:
# Visualize what the residual connection does
with torch.no_grad():
    attn_only, _ = block.attn(block.norm1(x))
    after_residual = x + attn_only

# Cosine similarity between input and output (per word)
cos_sims = F.cosine_similarity(x, after_residual, dim=-1)

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(range(SEQ_LEN), cos_sims.numpy(), color=TEAL, edgecolor='white', linewidth=1.5)
ax.set_xticks(range(SEQ_LEN))
ax.set_xticklabels(words, fontsize=11, fontweight='bold')
ax.set_ylabel('Cosine Similarity (input → output)', fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_title('Residual Connection: Input vs Output of One Block', fontsize=13, color=ACCENT, fontweight='bold')
for i, v in enumerate(cos_sims.numpy()):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=9)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

print("Residual connections mean each word's output ≈ input + small update.")
print("This lets gradients flow through many stacked blocks without vanishing.")
print("Without residuals, deep transformers (12+ layers) can't train at all.")

---
## Step 5: The Full Transformer

Stack N blocks. Each block refines every token's representation:
- **Early layers:** low-level patterns (syntax, word grouping)
- **Later layers:** high-level semantics (who did what to whom, intent)

In [ ]:
class Transformer(nn.Module):
    """Full Transformer encoder: embedding + positional encoding + N blocks."""
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=6, d_ff=512):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, token_ids):
        # token_ids: [batch, seq_len]
        x = self.embed(token_ids) + self.pos_enc(token_ids.shape[1])
        for block in self.blocks:
            x = block(x)           # each block refines every token
        return self.final_norm(x)  # [batch, seq_len, d_model]


transformer = Transformer(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_heads=4,
    n_layers=6,
    d_ff=512,
)

total_params = sum(p.numel() for p in transformer.parameters())
print(f"Full Transformer:")
print(f"  Layers: 6")
print(f"  d_model: {D_MODEL}, n_heads: 4, d_ff: 512")
print(f"  Total parameters: {total_params:,}")
print(f"\nFor comparison:")
print(f"  GPT-2 Small:  124M params (12 layers, d=768)")
print(f"  PaLM (Gemma): 2B+ params (18+ layers, d=2048)")
print(f"  Our toy:      {total_params / 1e6:.1f}M params (6 layers, d={D_MODEL})")

In [ ]:
# Run the full transformer on our sentence
with torch.no_grad():
    output = transformer(token_ids)  # [1, seq_len, d_model]

print(f"Input:  token_ids {list(token_ids.shape)} → {SEQ_LEN} words")
print(f"Output: {list(output.shape)} → {SEQ_LEN} vectors of {D_MODEL}-d each")
print(f"\nKey difference from GRU:")
print(f"  GRU output:         [batch, {D_MODEL}]  — ONE vector for the whole sentence")
print(f"  Transformer output: [batch, {SEQ_LEN}, {D_MODEL}]  — one vector PER WORD")
print(f"\nEach word's output encodes its meaning contextualized by ALL other words.")
print(f"No bottleneck — 'cup' at position 7 carries the full context:")
print(f"  'the red cup that was picked up from the table'")

---
## Step 6: Transformer vs GRU — Head to Head

Let's encode the **same sentences** with both a GRU and our Transformer, and compare the outputs directly.

In [ ]:
# Build a GRU encoder (same as Language_Encoding_Levels notebook)
class TextEncoderGRU(nn.Module):
    def __init__(self, vocab_size, d_word=128, d_model=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_word)
        self.gru = nn.GRU(d_word, d_model, batch_first=True)
        self.ln = nn.LayerNorm(d_model)

    def forward(self, token_ids):
        x = self.embed(token_ids)       # [batch, seq_len, d_word]
        _, h_last = self.gru(x)         # h_last: [1, batch, d_model]
        return self.ln(h_last[0])       # [batch, d_model]

gru_encoder = TextEncoderGRU(VOCAB_SIZE, d_word=D_MODEL, d_model=D_MODEL)
gru_params = sum(p.numel() for p in gru_encoder.parameters())

print(f"Architecture comparison:")
print(f"  GRU:         {gru_params:>10,} params → output: [batch, {D_MODEL}]")
print(f"  Transformer: {total_params:>10,} params → output: [batch, seq_len, {D_MODEL}]")

In [ ]:
# Encode the same sentence with both
with torch.no_grad():
    gru_out = gru_encoder(token_ids)        # [1, d_model]
    tfm_out = transformer(token_ids)         # [1, seq_len, d_model]

gru_vec = gru_out[0].numpy()                 # [d_model]
tfm_vecs = tfm_out[0].numpy()                # [seq_len, d_model]

# Visualize: GRU = 1 vector, Transformer = N vectors
fig, axes = plt.subplots(2, 1, figsize=(14, 6), gridspec_kw={'height_ratios': [1, 4]})

# GRU: single vector
ax = axes[0]
im0 = ax.imshow(gru_vec.reshape(1, -1)[:, :64], aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
ax.set_yticks([0])
ax.set_yticklabels(['Entire sentence'], fontsize=11)
ax.set_xlabel('Hidden dimensions (first 64)', fontsize=10)
ax.set_title(f'GRU Output: ONE {D_MODEL}-d vector for "{sentence}"',
             fontsize=13, color=ACCENT, fontweight='bold')

# Transformer: per-token vectors
ax = axes[1]
im1 = ax.imshow(tfm_vecs[:, :64], aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
ax.set_yticks(range(SEQ_LEN))
ax.set_yticklabels(words, fontsize=11)
ax.set_xlabel('Hidden dimensions (first 64)', fontsize=10)
ax.set_title(f'Transformer Output: {SEQ_LEN} × {D_MODEL}-d vectors (one per word)',
             fontsize=13, color=TEAL, fontweight='bold')

plt.tight_layout()
plt.show()

print("GRU: The ENTIRE sentence is compressed into a single row.")
print("  → Early words ('the', 'robot') may have faded from the hidden state.")
print(f"\nTransformer: Each word gets its OWN {D_MODEL}-d vector.")
print("  → 'cup' carries full context: 'the RED cup picked up from the TABLE'")
print("  → 'picked' knows it connects to 'robot' (subject) and 'cup' (object)")

In [ ]:
# GRU hidden state evolution vs Transformer per-token similarity
# Show: GRU forgets early words, Transformer preserves all of them

# Trace GRU hidden states word by word
gru_hidden_states = []
h = None
with torch.no_grad():
    for t in range(SEQ_LEN):
        word_emb = gru_encoder.embed(token_ids[:, t:t+1])  # [1, 1, d_model]
        _, h = gru_encoder.gru(word_emb, h)
        gru_hidden_states.append(h[0, 0].numpy().copy())

gru_states = np.array(gru_hidden_states)  # [seq_len, d_model]

# Pairwise cosine similarity: GRU states vs Transformer outputs
def cosine_sim_matrix(vecs):
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1, norms)
    normed = vecs / norms
    return normed @ normed.T

gru_sim = cosine_sim_matrix(gru_states)
tfm_sim = cosine_sim_matrix(tfm_vecs)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, sim, title, c in [
    (axes[0], gru_sim, 'GRU: Hidden States After Each Word', ACCENT),
    (axes[1], tfm_sim, 'Transformer: Per-Token Outputs', TEAL),
]:
    im = ax.imshow(sim, cmap='RdYlGn', vmin=-1, vmax=1)
    ax.set_xticks(range(SEQ_LEN))
    ax.set_yticks(range(SEQ_LEN))
    ax.set_xticklabels(words, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(words, fontsize=9)
    for i in range(SEQ_LEN):
        for j in range(SEQ_LEN):
            val = sim[i, j]
            color = 'white' if abs(val) > 0.5 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7, color=color)
    ax.set_title(title, fontsize=12, color=c, fontweight='bold')

plt.suptitle('GRU vs Transformer: How Do Token Representations Relate?',
             fontsize=15, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

print("GRU (left): Later hidden states dominate — early words get overwritten.")
print("  Adjacent states are very similar (high similarity near diagonal).")
print("  State after 'table' has almost forgotten 'the robot picked'.")
print("\nTransformer (right): Each word has its own distinct representation.")
print("  Similarity reflects MEANING, not position. Related words can be similar")
print("  even when far apart (after training).")

---
## Step 7: The Computational Cost — O(N²)

Attention computes a score for **every pair** of tokens. This is powerful but expensive.

In [ ]:
import time

# Measure attention cost at different sequence lengths
seq_lengths = [10, 50, 100, 200, 500, 1000]
times_attn = []

test_mha = MultiHeadAttention(d_model=D_MODEL, n_heads=4)
test_mha.eval()

for n in seq_lengths:
    x_test = torch.randn(n, D_MODEL)
    # Warm up
    with torch.no_grad():
        _ = test_mha(x_test)
    # Time it
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(10):
            _ = test_mha(x_test)
    elapsed = (time.perf_counter() - start) / 10
    times_attn.append(elapsed * 1000)  # ms

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Attention scores count (N²)
ax = axes[0]
scores_count = [n * n for n in seq_lengths]
ax.plot(seq_lengths, scores_count, 'o-', color=ACCENT, linewidth=2, markersize=8)
ax.set_xlabel('Sequence Length (N)', fontsize=11)
ax.set_ylabel('Attention Scores (N²)', fontsize=11)
ax.set_title('Attention is O(N²)', fontsize=13, color=ACCENT, fontweight='bold')
ax.grid(True, alpha=0.3)
for n, s in zip(seq_lengths, scores_count):
    if s > 1000:
        ax.annotate(f'{s:,}', (n, s), textcoords='offset points',
                    xytext=(5, 10), fontsize=9, color=ACCENT)

# Right: Actual timing
ax = axes[1]
ax.plot(seq_lengths, times_attn, 's-', color=BLUE, linewidth=2, markersize=8)
ax.set_xlabel('Sequence Length (N)', fontsize=11)
ax.set_ylabel('Time per Forward Pass (ms)', fontsize=11)
ax.set_title('Wall-Clock Time (CPU)', fontsize=13, color=BLUE, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.suptitle('The Cost of Seeing Everything: O(N²) Attention',
             fontsize=15, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

print("The N² cost is why long sequences (10K+ tokens) are expensive.")
print("But for robot instructions (~10-50 tokens), it's negligible.")
print("For VLAs: image patches (~200) + text tokens (~50) = ~250 tokens → manageable.")

---
## Step 8: How Transformers Fix Our Three Failures

Let's make each fix concrete with code.

In [ ]:
# Fix 1: No more bottleneck — N vectors instead of 1
print("="*60)
print("FIX 1: NO MORE BOTTLENECK")
print("="*60)

with torch.no_grad():
    gru_output = gru_encoder(token_ids)   # [1, 128]
    tfm_output = transformer(token_ids)    # [1, 10, 128]

print(f"\nGRU output:         {list(gru_output.shape)}")
print(f"  → 1 vector for '{sentence}'")
print(f"  → ALL information compressed into 128 numbers")
print(f"  → 'the' and 'robot' at the start may be forgotten")

print(f"\nTransformer output: {list(tfm_output.shape)}")
print(f"  → {SEQ_LEN} vectors, one per word")
print(f"  → 'cup' at position 7 has full context of the WHOLE sentence")
print(f"  → Can pool (mean/cls) if you need a single vector, but information is preserved")

In [ ]:
# Fix 2: Cross-modal attention — vision + language in the same sequence
print("="*60)
print("FIX 2: CROSS-MODAL ATTENTION")
print("="*60)

# Simulate: 4 image patch tokens + 5 text tokens in the same sequence
n_image_patches = 4
n_text_tokens = 5
n_total = n_image_patches + n_text_tokens

# Random "image patch" embeddings + random "text" embeddings
fake_image_tokens = torch.randn(n_image_patches, D_MODEL) * 0.5
fake_text_tokens = torch.randn(n_text_tokens, D_MODEL) * 0.5
combined = torch.cat([fake_image_tokens, fake_text_tokens], dim=0)  # [9, 128]

# Run through one transformer block
with torch.no_grad():
    block_out = block(combined)

    # Check: attention between image and text tokens
    _, head_weights = block.attn(block.norm1(combined))

# Visualize cross-modal attention in head 0
w = head_weights[0].detach().numpy()
token_labels = [f'img_{i}' for i in range(n_image_patches)] + \
               ['push', 'the', 'block', 'to', 'right']

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(w, cmap='YlOrRd', vmin=0)
ax.set_xticks(range(n_total))
ax.set_yticks(range(n_total))
ax.set_xticklabels(token_labels, rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(token_labels, fontsize=10)

for i in range(n_total):
    for j in range(n_total):
        val = w[i, j]
        color = 'white' if val > 0.15 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8, color=color)

# Draw boxes around cross-modal regions
from matplotlib.patches import Rectangle
ax.add_patch(Rectangle((-0.5, n_image_patches - 0.5), n_image_patches, n_text_tokens,
                        fill=False, edgecolor=ACCENT, linewidth=3, linestyle='--'))
ax.add_patch(Rectangle((n_image_patches - 0.5, -0.5), n_text_tokens, n_image_patches,
                        fill=False, edgecolor=BLUE, linewidth=3, linestyle='--'))
ax.text(1.5, 6.5, 'text→image', fontsize=10, color=ACCENT, fontweight='bold', ha='center')
ax.text(6.5, 1.5, 'image→text', fontsize=10, color=BLUE, fontweight='bold', ha='center')

plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Cross-Modal Attention: Image + Text in One Sequence',
             fontsize=13, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

print('Dashed boxes show CROSS-MODAL attention:')
print('  Orange: text tokens attending to image patches')
print('  Blue: image patches attending to text tokens')
print("\nThis is impossible with the GRU + concat approach from Part 2.")
print("The GRU encodes text in isolation — 'red' never sees any pixels.")
print("With Transformers, 'red' directly attends to red image patches!")

In [ ]:
# Fix 3: Parallel processing
print("="*60)
print("FIX 3: PARALLEL PROCESSING")
print("="*60)

# Time GRU (sequential) vs Transformer (parallel)
test_ids = torch.randint(0, VOCAB_SIZE, (1, 50))  # 50 tokens

# Need matching architectures for fair comparison
gru_test = TextEncoderGRU(VOCAB_SIZE, d_word=D_MODEL, d_model=D_MODEL)
tfm_test = Transformer(VOCAB_SIZE, d_model=D_MODEL, n_heads=4, n_layers=6, d_ff=512)
gru_test.eval()
tfm_test.eval()

# GRU: must process sequentially
start = time.perf_counter()
with torch.no_grad():
    for _ in range(50):
        _ = gru_test(test_ids)
gru_time = (time.perf_counter() - start) / 50 * 1000

# Transformer: all positions in parallel
start = time.perf_counter()
with torch.no_grad():
    for _ in range(50):
        _ = tfm_test(test_ids)
tfm_time = (time.perf_counter() - start) / 50 * 1000

print(f"\n50-token sequence, CPU (averaged over 50 runs):")
print(f"  GRU (sequential):    {gru_time:.2f} ms")
print(f"  Transformer (parallel): {tfm_time:.2f} ms")
print(f"\nThe Transformer has MORE parameters but attention is a single matrix multiply.")
print(f"On GPU, this parallelism makes Transformers MUCH faster than GRUs for long sequences.")
print(f"\nThe GRU MUST process word 5 before word 6 (hidden state dependency).")
print(f"The Transformer computes ALL pairwise scores in one N×N matrix multiply.")

---
## Summary

| Component | What it does | Key insight |
|-----------|-------------|-------------|
| **Q, K, V** | Three projections of each word | Query = "what do I need?", Key = "what do I have?", Value = "my content" |
| **Scaled Dot-Product** | Q·K^T / √d_k | Without scaling, softmax becomes one-hot |
| **Multi-Head** | N parallel attention heads | Each head learns a different relationship pattern |
| **Positional Encoding** | Sinusoidal position signal | Without it, attention is order-blind |
| **FFN** | Expand 4×, GELU, compress | Per-token "thinking" after attention mixing |
| **Residual + LayerNorm** | x + sublayer(x) | Enables training deep (100+ layer) networks |
| **Stacked Blocks** | N blocks in sequence | Early = syntax, Late = semantics |

### How This Fixes the mini-VLA Failures

| Failure | GRU (Part 2) | Transformer (Part 3) |
|---------|-------------|---------------------|
| Bottleneck | 1 vector for whole sentence | N vectors — one per token |
| No cross-modal | Separate encoders, MLP concat | All tokens in one sequence — cross-attention |
| Sequential | Must read word by word | All pairs computed in parallel (matrix multiply) |

In **Part 4**, we'll see how real VLAs (RT-2, OpenVLA, pi0) use pre-trained Transformers (ViT for vision, Gemma/LLaMA for language) to build powerful robot policies.

---

## Exercises

1. **Change n_heads:** Try 1, 2, 8, 16 heads with d_model=128. What happens to the attention maps? How does the parameter count change?
2. **Remove residual connections:** Comment out the `x + ...` in `TransformerBlock.forward()`. Does the model still produce diverse per-token outputs after 6 layers, or do they all collapse to the same vector?
3. **Remove positional encoding:** Remove the `+ self.pos_enc(...)` line in `Transformer.forward()`. Verify that shuffled input produces identical attention patterns to the original.
4. **Causal masking:** Add a causal mask to attention (mask future tokens). This is what GPT and language models use. Hint: add `-1e9` to the upper triangle of the scores matrix before softmax.
5. **Challenge:** Replace the GRU text encoder in the mini-VLA (Build_MiniVLA.ipynb) with a small Transformer. Does it learn faster? Does it generalize better?